## Data Preparation

### 1. Konfigurasi Modul

In [2]:
import sys
import os
import cv2
import albumentations as A
import numpy as np
from glob import glob
from tqdm import tqdm
from PIL import Image
from pillow_heif import register_heif_opener
import mediapipe as mp

# 1. Konfigurasi Path dan Parameter
INPUT_FOLDER = '1_raw_dataset'      
OUTPUT_FOLDER = '2_processed_dataset'   
AUG_PER_IMAGE = 5                      
TARGET_SIZE = (512, 512)
PADDING_PCT = 0.2                      

register_heif_opener()

# 2. Nyalakan AI Pendeteksi Tangan Menggunakan API Terbaru ('tasks')
try:
    # Memanfaatkan 'tasks' yang terbukti ada di dalam modultmu
    BaseOptions = mp.tasks.BaseOptions
    HandLandmarker = mp.tasks.vision.HandLandmarker
    HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
    VisionRunningMode = mp.tasks.vision.RunningMode

    # Mengunduh model pendeteksi tangan resmi dari Google secara otomatis jika belum ada
    model_path = 'hasil_data_preparation/hand_landmarker.task'
    if not os.path.exists(model_path):
        print("Sedang mengunduh file pendeteksi tangan dari Google (hanya sekali)...")
        import urllib.request
        url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task"
        urllib.request.urlretrieve(url, model_path)

    options = HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=model_path),
        running_mode=VisionRunningMode.IMAGE,
        num_hands=1,
        min_hand_detection_confidence=0.5
    )
    
    landmarker = HandLandmarker.create_from_options(options)
    print("✓ LUAR BIASA! MediaPipe Hand Landmarker (API Tasks Terbaru) Berhasil Dinyalakan!")
except Exception as e:
    print(f"\n FATAL ERROR UTAMA API TASKS: {e}")
    sys.exit()

# 3. Fungsi Pemotong Tangan (Hand Cropping) Sesuai API Baru
def get_hand_crop(image_rgb, padding=0.2):
    height, width, _ = image_rgb.shape
    
    # Bungkus gambar ke format internal MediaPipe Image
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
    results = landmarker.detect(mp_image)

    if results.hand_landmarks:
        hand_landmarks = results.hand_landmarks[0]
        x_list = [lm.x for lm in hand_landmarks]
        y_list = [lm.y for lm in hand_landmarks]
        
        x_min, x_max = int(min(x_list) * width), int(max(x_list) * width)
        y_min, y_max = int(min(y_list) * height), int(max(y_list) * height)
        
        box_w = x_max - x_min
        box_h = y_max - y_min
        
        pad_x = int(box_w * padding)
        pad_y = int(box_h * padding)
        
        new_x_min = max(0, x_min - pad_x)
        new_y_min = max(0, y_min - pad_y)
        new_x_max = min(width, x_max + pad_x)
        new_y_max = min(height, y_max + pad_y)
        
        return image_rgb[new_y_min:new_y_max, new_x_min:new_x_max]
    return None

# 4. Racikan Resep Augmentasi (Albumentations)
transform = A.Compose([
    A.LongestMaxSize(max_size=TARGET_SIZE[0]),
    A.PadIfNeeded(min_height=TARGET_SIZE[0], min_width=TARGET_SIZE[1], border_mode=cv2.BORDER_CONSTANT, value=0, p=1.0),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.15, rotate_limit=0, border_mode=cv2.BORDER_CONSTANT, value=0, p=0.8),
    A.OneOf([A.RandomBrightnessContrast(p=1), A.GaussNoise(p=1), A.MotionBlur(p=1)], p=0.5),
    A.CoarseDropout(max_holes=10, max_height=15, max_width=15, min_holes=5, fill_value=0, p=0.5),
])

Sedang mengunduh file pendeteksi tangan dari Google (hanya sekali)...
✓ LUAR BIASA! MediaPipe Hand Landmarker (API Tasks Terbaru) Berhasil Dinyalakan!


I0000 00:00:1780505853.833731    6558 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1780505853.836294    6570 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.0.7-0ubuntu0.24.04.2), renderer: Mesa Intel(R) UHD Graphics (ICL GT1)
W0000 00:00:1780505853.857692    6563 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1780505853.870913    6565 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/tmp/ipykernel_5711/161249579.py:83: UserWarning: Argument(s) 'value' are not valid for transform PadIfNeeded
  A.PadIfNeeded(min_height=TARGET_SIZE[0], min_width=TARGET_SIZE[1], border_mode=cv2.BORDER_CONSTANT, value=0, p=1.0),
/tmp/ipykernel_5711/161249579.py:84: UserWarning: Argument(s) 'value' are not valid for transform ShiftScaleRotate
  A.ShiftScaleRot

### 2. Transformasi Citra

In [3]:
# ==========================================
# CELL 2: EKSEKUSI UTAMA (CROP & AUGMENTASI)
# ==========================================

# 1. Cari semua file gambar di folder raw_dataset
valid_extensions = ('.jpg', '.jpeg', '.png', '.heic', '.HEIC')
image_paths = [f for f in glob(os.path.join(INPUT_FOLDER, '**', '*.*'), recursive=True) if f.lower().endswith(valid_extensions)]

print(f"MULAI PROSES SIBI!! Menemukan {len(image_paths)} gambar asli di folder '{INPUT_FOLDER}'.")

# 2. Siapkan variabel pencatat statistik skripsi
success_count = 0
skipped_list = []  
error_list = []    
class_counts = {} 

# 3. Mulai Perulangan Pemrosesan (Crop Dulu -> Baru Augmentasi)
for img_path in tqdm(image_paths):
    try:
        # A. Buka Gambar Asli
        pil_img = Image.open(img_path).convert("RGB")
        image_np = np.array(pil_img)
        
        # B. LANGKAH 1: Potong fokus tangan (Menggunakan model Tasks dari Cell 1)
        hand_img = get_hand_crop(image_np, padding=PADDING_PCT)
        if hand_img is None:
            skipped_list.append(img_path)
            continue  # Jika tangan tidak terdeteksi, lewati ke gambar berikutnya

        # C. Manajemen Penamaan Folder Target (A-Z)
        rel_path = os.path.relpath(img_path, INPUT_FOLDER)
        subfolder = os.path.dirname(rel_path)
        base_name = os.path.splitext(os.path.basename(img_path))[0]
        
        class_name = os.path.basename(subfolder)
        if not class_name: class_name = "Uncategorized"

        target_dir = os.path.join(OUTPUT_FOLDER, subfolder)
        os.makedirs(target_dir, exist_ok=True)

        # D. LANGKAH 2: Beri Efek Albumentations pada Hasil Potongan Tangan
        for i in range(AUG_PER_IMAGE):
            augmented = transform(image=hand_img)['image']
            
            # Konversi kembali ke format BGR agar bisa disimpan dengan OpenCV
            save_img = cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR)
            save_path = os.path.join(target_dir, f"{base_name}_v3_{i}.jpg")
            cv2.imwrite(save_path, save_img)
            success_count += 1

            # Catat jumlah data per kelas untuk laporan Bab 4
            if class_name in class_counts:
                class_counts[class_name] += 1
            else:
                class_counts[class_name] = 1
            
    except Exception as e:
        error_list.append((img_path, str(e)))

# 4. CETAK RINGKASAN LAPORAN UNTUK DATASET BARU
print("\n" + "="*50)
print("DISTRIBUSI KELAS DATASET BARU (HASIL FOCUSED & AUGMENTED)")
print("="*50)
for cls_name in sorted(class_counts.keys()):
    print(f" Folder Kelas '{cls_name}' : {class_counts[cls_name]} gambar baru")

print("\n" + "="*50)
print("RINGKASAN AKHIR PROSES")
print("="*50)
print(f" TOTAL BERHASIL GENERATE : {success_count} file gambar baru")
print(f" TOTAL DI-SKIP (No Hand) : {len(skipped_list)} file asli")
print(f" TOTAL FILE RUSAK/ERROR  : {len(error_list)} file asli")
print("="*50)

# Jika ada file yang gagal terdeteksi tangannya, tampilkan jalurnya di sini
if len(skipped_list) > 0:
    print(f"\n💡 INFO: Ada {len(skipped_list)} gambar di-skip karena posisi/pencahayaan tangan kurang terbaca oleh MediaPipe.")

MULAI PROSES SIBI!! Menemukan 637 gambar asli di folder '1_raw_dataset'.


100%|██████████| 637/637 [02:21<00:00,  4.50it/s]


DISTRIBUSI KELAS DATASET BARU (HASIL FOCUSED & AUGMENTED)
 Folder Kelas 'A' : 225 gambar baru
 Folder Kelas 'B' : 225 gambar baru
 Folder Kelas 'C' : 225 gambar baru
 Folder Kelas 'D' : 210 gambar baru
 Folder Kelas 'E' : 75 gambar baru
 Folder Kelas 'F' : 100 gambar baru
 Folder Kelas 'G' : 100 gambar baru
 Folder Kelas 'H' : 95 gambar baru
 Folder Kelas 'I' : 100 gambar baru
 Folder Kelas 'J' : 100 gambar baru
 Folder Kelas 'K' : 100 gambar baru
 Folder Kelas 'L' : 100 gambar baru
 Folder Kelas 'M' : 100 gambar baru
 Folder Kelas 'N' : 100 gambar baru
 Folder Kelas 'O' : 120 gambar baru
 Folder Kelas 'P' : 85 gambar baru
 Folder Kelas 'Q' : 100 gambar baru
 Folder Kelas 'R' : 120 gambar baru
 Folder Kelas 'S' : 100 gambar baru
 Folder Kelas 'T' : 170 gambar baru
 Folder Kelas 'U' : 100 gambar baru
 Folder Kelas 'V' : 100 gambar baru
 Folder Kelas 'W' : 100 gambar baru
 Folder Kelas 'X' : 100 gambar baru
 Folder Kelas 'Y' : 100 gambar baru
 Folder Kelas 'Z' : 75 gambar baru

RINGKASA

### 3. Otomatisasi Pelabelan Dataset dan Pembuatan Manifest CSV

In [ ]:
import os
import pandas as pd
import pprint

# ==========================================
# CELL 3: OTOMATISASI PELABELAN DATASET
# ==========================================

def proses_pelabelan_sibi_terurut(data_dir):
    """
    Fungsi pelabelan data SIBI yang diperbaiki agar indeks kelas (0-25)
    sinkron secara alfabetis dengan folder huruf A sampai Z.
    """
    input_fitur_gambar = []
    label_target_kelas = []
    peta_kelas = {} 
    
    if os.path.exists(data_dir):
        # 1. Ambil semua item di dalam direktori dan filter hanya yang berupa folder
        semua_item = os.listdir(data_dir)
        daftar_folder = [f for f in semua_item if os.path.isdir(os.path.join(data_dir, f))]
        
        # 2. Paksa urutkan folder secara alfabetis (A, B, C, D... sampai Z)
        daftar_folder_urut = sorted(daftar_folder)
        
        # 3. Proses pelabelan berurutan berdasarkan folder yang sudah rapi
        for indeks_kelas, nama_folder in enumerate(daftar_folder_urut):
            peta_kelas[nama_folder] = indeks_kelas 
            path_folder = os.path.join(data_dir, nama_folder)
            
            # Mencatat folder + nama file dari DATASET BARU hasil augmentasi
            for file_nama in os.listdir(path_folder):
                if file_nama.lower().endswith(('.jpg', '.jpeg', '.png', '.heic')):
                    # Menggabungkan nama folder huruf dan nama file (Contoh: "A/foto_v3_0.jpg")
                    path_relatif = os.path.join(nama_folder, file_nama) 
        
                    input_fitur_gambar.append(path_relatif)
                    label_target_kelas.append(indeks_kelas)
                    
        print(f"✓ Berhasil memetakan {len(label_target_kelas)} data gambar SIBI ke dalam {len(daftar_folder_urut)} kelas target.\n")
        return input_fitur_gambar, label_target_kelas, peta_kelas
    else:
        print(f"❌ Direktori '{data_dir}' tidak ditemukan. Periksa kembali path folder Anda.")
        return [], [], {}

# =====================================================================
# LINGKUNGAN EKSEKUSI (RUN BAGIAN INI)
# =====================================================================

# 1. PATH DIUBAH: Menembak folder output dari CELL 2 (Dataset Ter-augmentasi)
path_sesuai_gambar = '2_processed_dataset'

# 2. Jalankan fungsi pelabelan data
X_data, y_label, peta_kelas = proses_pelabelan_sibi_terurut(path_sesuai_gambar)

# 3. CETAK OUTPUT LOG PEMETAAN SECARA UTUH BARIS PER BARIS
if peta_kelas:
    print("=" * 50)
    print("      LOG PEMETAAN KELAS TARGET SIBI SECARA UTUH      ")
    print("=" * 50)
    for huruf, indeks in peta_kelas.items():
        print(f" Folder '{huruf}' ➔ Ditransformasikan ke Kelas {indeks}")
    print("=" * 50)

# 4. CETAK BEBERAPA SAMPEL DATA UNTUK VERIFIKASI SINKRONISASI
if X_data and y_label:
    print("\n[VERIFIKASI] Mengintip Hasil Pasangan Data di RAM:")
    print(f" - 5 Sampel Gambar Pertama : {X_data[:5]}")
    print(f" - 5 Sampel Label Pertama  : {y_label[:5]}")
    print(f" - 5 Sampel Gambar Terakhir: {X_data[-5:]}")
    print(f" - 5 Sampel Label Terakhir : {y_label[-5:]}\n")

# 5. SIMPAN HASIL PELABELAN MENJADI FILE CSV 
if X_data and y_label:
    df_label = pd.DataFrame({
        'Nama_File_Gambar': X_data,
        'Label_Kelas_Angka': y_label
    })
    nama_file_csv = 'hasil_data_preparation/hasil_pelabelan_sibi.csv'
    df_label.to_csv(nama_file_csv, index=False)
    print(f"✓ Sukses Ekspor! File '{nama_file_csv}' telah tersimpan di direktori utama.")

✓ Berhasil memetakan 3125 data gambar SIBI ke dalam 26 kelas target.

      LOG PEMETAAN KELAS TARGET SIBI SECARA UTUH      
 Folder 'A' ➔ Ditransformasikan ke Kelas 0
 Folder 'B' ➔ Ditransformasikan ke Kelas 1
 Folder 'C' ➔ Ditransformasikan ke Kelas 2
 Folder 'D' ➔ Ditransformasikan ke Kelas 3
 Folder 'E' ➔ Ditransformasikan ke Kelas 4
 Folder 'F' ➔ Ditransformasikan ke Kelas 5
 Folder 'G' ➔ Ditransformasikan ke Kelas 6
 Folder 'H' ➔ Ditransformasikan ke Kelas 7
 Folder 'I' ➔ Ditransformasikan ke Kelas 8
 Folder 'J' ➔ Ditransformasikan ke Kelas 9
 Folder 'K' ➔ Ditransformasikan ke Kelas 10
 Folder 'L' ➔ Ditransformasikan ke Kelas 11
 Folder 'M' ➔ Ditransformasikan ke Kelas 12
 Folder 'N' ➔ Ditransformasikan ke Kelas 13
 Folder 'O' ➔ Ditransformasikan ke Kelas 14
 Folder 'P' ➔ Ditransformasikan ke Kelas 15
 Folder 'Q' ➔ Ditransformasikan ke Kelas 16
 Folder 'R' ➔ Ditransformasikan ke Kelas 17
 Folder 'S' ➔ Ditransformasikan ke Kelas 18
 Folder 'T' ➔ Ditransformasikan ke Kelas 19
 Fold

### 4. Pembagian Dataset SIBI

In [5]:
import os
import splitfolders

# ==========================================
# CELL 4: PEMBAGIAN DATASET (TRAIN & VALIDATION SPLIT)
# ==========================================

# 1. Tentukan nama sub-judul berbasis machine learning
print("### 4. Pembagian Dataset SIBI Menggunakan Rasio Komparatif\n")

# 2. Definisikan folder input dan output final
DATASET_SUMBER = '2_processed_dataset'  # Hasil dari Cell 2
DATASET_FINAL = '3_final_dataset'       # Target akhir untuk training

if not os.path.exists(DATASET_FINAL):
    print(f"Sedang membagi data dari '{DATASET_SUMBER}' menjadi Train & Validation...")
    # Membagi dengan rasio 80% Training dan 20% Validation
    splitfolders.ratio(DATASET_SUMBER, output=DATASET_FINAL, seed=42, ratio=(.8, .2))
    print("✓ Pembagian data berhasil! Folder '3_final_dataset' siap digunakan.")
else:
    print(f"✓ Folder '{DATASET_FINAL}' sudah ada. Dataset siap digunakan untuk training.")

### 4. Pembagian Dataset SIBI Menggunakan Rasio Komparatif

✓ Folder '3_final_dataset' sudah ada. Dataset siap digunakan untuk training.


### 5. Transformasi Matriks Citra & Kategorisasi Target Kelas

In [6]:
import tensorflow as tf
from tensorflow import keras
# Memanggil ImageDataGenerator langsung lewat pembungkus utama
ImageDataGenerator = tf.keras.preprocessing.image.ImageDataGenerator

# ==========================================
# CELL 5: KONSTRUKSI DATA GENERATOR & ENCODING
# ==========================================

print("### 5. Standardisasi Dimensi, Normalisasi Piksel, dan One-Hot Encoding\n")

# 1. Definisikan Path Folder Hasil Split
train_dir = os.path.join('3_final_dataset', 'train')
val_dir = os.path.join('3_final_dataset', 'val')

# 2. Normalisasi Data (Rescale 1./255) tanpa augmentasi tambahan
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)

# 3. Load Data Sekaligus Eksekusi Resizing & One-Hot Encoding
print("Memuat Dataset Training:")
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224), # Menyesuaikan resolusi input arsitektur MobileNetV2
    batch_size=32,
    class_mode='categorical' # Otomatis mengeksekusi konsep One-Hot Encoding
)

print("\nMemuat Dataset Validation:")
validation_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical' # Otomatis mengeksekusi konsep One-Hot Encoding
)

print("\n✓ SELAMAT! Seluruh tahap Data Preprocessing telah selesai 100%.")
print("Data Anda telah ternormalisasi, ter-encode, dan siap dimasukkan ke Model Training!")

### 5. Standardisasi Dimensi, Normalisasi Piksel, dan One-Hot Encoding

Memuat Dataset Training:
Found 2500 images belonging to 26 classes.

Memuat Dataset Validation:
Found 625 images belonging to 26 classes.

✓ SELAMAT! Seluruh tahap Data Preprocessing telah selesai 100%.
Data Anda telah ternormalisasi, ter-encode, dan siap dimasukkan ke Model Training!


In [7]:
import numpy as np

# 1. Definisikan matriks koefisien (A) dan vektor konstanta (B) dari neraca massa
A = np.array([[100, -20, 0],
              [-50, 140, -100],
              [-20, 0, 100]], dtype=float)

B = np.array([500, 0, 900], dtype=float)
n = len(B)

print("=====================================================================")

# =========================================================================
# 1. METODE ELIMINASI GAUSS
# =========================================================================
print("Sistem Perhitungan Konsentrasi Reaktor dengan Metode Gauss")
print("Hasil Eliminasi Gauss")

# Membuat matriks augmentasi [A|B]
AugGauss = np.hstack([A, B.reshape(-1, 1)])

# Proses Eliminasi Maju (Forward Elimination)
for i in range(n-1):
    for j in range(i+1, n):
        faktor = AugGauss[j, i] / AugGauss[i, i]
        AugGauss[j, i:] -= faktor * AugGauss[i, i:]

# Menampilkan hasil matriks eselon (segitiga atas) sesuai format gambar 2
print(f"       {AugGauss[0,0]:7.0f}        {AugGauss[0,1]:7.0f}        {AugGauss[0,2]:7.0f}        {AugGauss[0,3]:7.0f}")
print(f"       {AugGauss[1,0]:7.0f}        {AugGauss[1,1]:7.0f}        {AugGauss[1,2]:7.0f}        {AugGauss[1,3]:7.0f}")
print(f"       {AugGauss[2,0]:7.0f}        {AugGauss[2,1]:7.0f}        {AugGauss[2,2]:7.3f}        {AugGauss[2,3]:7.1f}")

# Substitusi Mundur (Back Substitution)
X_gauss = np.zeros(n)
X_gauss[n-1] = AugGauss[n-1, -1] / AugGauss[n-1, n-1]
for i in range(n-2, -1, -1):
    X_gauss[i] = (AugGauss[i, -1] - np.dot(AugGauss[i, i+1:n], X_gauss[i+1:n])) / AugGauss[i, i]

print("\nSolusi:")
for val in X_gauss:
    print(f"    {val:7.4f}")

print("=====================================================================")

# =========================================================================
# 2. METODE GAUSS JORDAN
# =========================================================================
print("Sistem Perhitungan Konsentrasi Reaktor dengan Metode Gauss Jordan")
print("Hasil Eliminasi Gauss Jordan")

AugGJ = np.hstack([A, B.reshape(-1, 1)])
for i in range(n):
    AugGJ[i] = AugGJ[i] / AugGJ[i, i]
    for j in range(n):
        if i != j:
            faktor = AugGJ[j, i]
            AugGJ[j] -= faktor * AugGJ[i]

# Menampilkan matriks identitas beserta solusi di kanannya (Gambar 3)
for row in AugGJ:
    print(f"    {row[0]:8.4f}    {row[1]:8.4f}    {row[2]:8.4f}    {row[3]:8.4f}")

print("\nSolusi:")
for val in AugGJ[:, -1]:
    print(f"    {val:7.4f}")

print("=====================================================================")

# =========================================================================
# 3. METODE INVERS MATRIKS
# =========================================================================
print("Sistem Perhitungan Konsentrasi Reaktor dengan Metode Invers Matriks")
print("Invers Matriks")

A_inv = np.linalg.inv(A)
# Menampilkan nilai matriks invers (Gambar 3 kanan)
for row in A_inv:
    print(f"    {row[0]:8.6f}    {row[1]:8.6f}    {row[2]:8.6f}")

X_inv = np.dot(A_inv, B)
print("\nSolusi:")
for val in X_inv:
    print(f"    {val:7.3f}")
print("=====================================================================")

Sistem Perhitungan Konsentrasi Reaktor dengan Metode Gauss
Hasil Eliminasi Gauss
           100            -20              0            500
             0            130           -100            250
             0              0         96.923         1007.7

Solusi:
     6.9841
     9.9206
    10.3968
Sistem Perhitungan Konsentrasi Reaktor dengan Metode Gauss Jordan
Hasil Eliminasi Gauss Jordan
      1.0000      0.0000      0.0000      6.9841
      0.0000      1.0000      0.0000      9.9206
      0.0000      0.0000      1.0000     10.3968

Solusi:
     6.9841
     9.9206
    10.3968
Sistem Perhitungan Konsentrasi Reaktor dengan Metode Invers Matriks
Invers Matriks
    0.011111    0.001587    0.001587
    0.005556    0.007937    0.007937
    0.002222    0.000317    0.010317

Solusi:
      6.984
      9.921
     10.397
